<a href="https://colab.research.google.com/github/vivek28n/Medical-RAG-Hallucination-Detection/blob/main/notebooks/Notebook_08_Pipeline_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 08 — Complete Medical RAG Pipeline Integration

## Objective

This notebook integrates the components developed in Notebooks 03–07
into a single reusable medical RAG pipeline.

### Integrated Components

1. Medical document retrieval
2. Grounded LLM answer generation
3. Hallucination detection
4. Confidence scoring
5. Claim-level consistency verification
6. Self-correction
7. Final answer verification

### Final Pipeline

User Question
→ Document Retrieval
→ Grounded Answer Generation
→ Hallucination Detection
→ Confidence Scoring
→ Claim Verification
→ Self-Correction if Required
→ Final Verification
→ Structured Response

The goal of this notebook is integration and validation,
not the development of new model components.

In [1]:
import os
import sys
import numpy as np

PROJECT_DIR = "/content/Medical-RAG-Hallucination-Detection"

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

%cd /content/Medical-RAG-Hallucination-Detection

print("Project directory:", os.getcwd())
print("Project exists:", os.path.exists(PROJECT_DIR))

/content/Medical-RAG-Hallucination-Detection
Project directory: /content/Medical-RAG-Hallucination-Detection
Project exists: True


In [2]:
!pip install -q pymupdf faiss-cpu langchain-text-splitters sentence-transformers transformers torch google-genai

In [3]:
import os
import re
import time
import numpy as np
import fitz
import faiss
import torch

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [4]:
# Cell 4 — Load Medical PDF and Build Retrieval Index

PDF_PATH = os.path.join(
    PROJECT_DIR,
    "dataset",
    "raw",
    "niddk_guiding_principles_diabetes.pdf"
)

print("PDF exists:", os.path.exists(PDF_PATH))
print("PDF path:", PDF_PATH)

PDF exists: True
PDF path: /content/Medical-RAG-Hallucination-Detection/dataset/raw/niddk_guiding_principles_diabetes.pdf


In [5]:
# Cell 5 — Extract, Clean, Chunk and Index Medical PDF

import pymupdf

# Open PDF
doc = pymupdf.open(PDF_PATH)

print("Total pages:", len(doc))

# Extract and clean text
pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text("text")
    text = re.sub(r"\s+", " ", text).strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

doc.close()

print("Pages with text:", len(pages))

# Create chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page_data in pages:
    page_chunks = text_splitter.split_text(page_data["text"])

    for chunk_id, chunk_text in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page_data['page']}_chunk_{chunk_id}",
            "page": page_data["page"],
            "text": chunk_text
        })

print("Total chunks:", len(chunks))

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

# Build FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS vectors:", index.ntotal)
print("Embedding dimension:", embeddings.shape[1])

Total pages: 83
Pages with text: 83
Total chunks: 268


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding shape: (268, 384)
FAISS vectors: 268
Embedding dimension: 384


In [6]:
# Cell 6 — Medical Document Retrieval

def retrieve_documents(question, top_k=5):
    """
    Retrieve the most relevant medical document chunks
    for a given question using FAISS.
    """

    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "source": "NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes",
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distances[0][rank - 1])
        })

    return results


# Test retrieval
test_question = "What are the risk factors for type 2 diabetes?"

retrieved_docs = retrieve_documents(
    test_question,
    top_k=5
)

print("Retrieved documents:", len(retrieved_docs))

for doc_item in retrieved_docs:
    print(
        f"\nRank {doc_item['rank']} | "
        f"Page {doc_item['page']} | "
        f"Distance {doc_item['distance']:.4f}"
    )
    print(doc_item["text"][:300], "...")

Retrieved documents: 5

Rank 1 | Page 5 | Distance 0.6296
5 INTRODUCTION The diabetes problem Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes also increases the risk of cardiovascular disease, cancer, and dementia a ...

Rank 2 | Page 5 | Distance 0.7152
diabetes have A1C > 8; 28 percent have BP > 140/90; 51 percent are on statins; 44 percent have LDL > 100; and 20 percent use tobacco. Thus, a substantial proportion of people with diabetes do not meet goals generally agreed as appropriate for most individuals with diabetes.5 The National Institutes  ...

Rank 3 | Page 5 | Distance 0.7260
by the Centers for Disease Control and Prevention (CDC) and other organizations. Proper nutrition and physical activity are the cornerstones of treatment and prevention of type 2 diabetes. In addition to lifestyle modifications and tobacco avoidance, controlling

In [7]:
# Cell 7 — Build Grounded Context

def build_context(retrieved_docs):
    context_parts = []

    for doc_item in retrieved_docs:
        context_parts.append(
            f"[Source: {doc_item['source']} | "
            f"Page: {doc_item['page']} | "
            f"Chunk: {doc_item['chunk_id']}]\n"
            f"{doc_item['text']}"
        )

    return "\n\n".join(context_parts)


context = build_context(retrieved_docs)

print(context[:3000])

[Source: NIDDK Guiding Principles for the Care of People with or at Risk for Diabetes | Page: 5 | Chunk: page_5_chunk_0]
5 INTRODUCTION The diabetes problem Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327 billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2 Another 84.1 million Americans (33.9 percent of adults) have glucose levels that are higher than normal but not high enough to be characterized as diabetes.1 Because persons with these glucose levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes by the Centers for Disease Control and Prevention (CDC) and other organizat

In [39]:
# Cell 8 — Grounded LLM Answer Generation

from google import genai
from google.colab import userdata

API_KEY = userdata.get("Vivek28n")

if not API_KEY:
    raise ValueError("Gemini API key not found in Colab Secrets.")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.8-flash"


def generate_grounded_answer(question, context, max_retries=3):

    prompt = f"""
You are a medical information assistant.

Answer the user's question using ONLY the provided medical context.

Rules:
1. Do not use outside knowledge.
2. Do not invent or assume facts.
3. If the context is insufficient, say that clearly.
4. Keep the answer evidence-based.
5. Preserve important medical qualifiers.
6. Do not provide diagnosis or personalized medical advice.
7. Mention the relevant source page for factual claims.

MEDICAL CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    last_error = None

    for attempt in range(1, max_retries + 1):

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            if response and response.text:
                return response.text.strip()

            last_error = "Gemini returned an empty response."

        except Exception as e:
            last_error = str(e)
            print(
                f"Attempt {attempt}/{max_retries} failed: "
                f"{last_error}"
            )

        if attempt < max_retries:
            time.sleep(4)

    raise RuntimeError(
        f"Gemini generation failed after {max_retries} attempts. "
        f"Last error: {last_error}"
    )


# Generate answer again
generated_answer = generate_grounded_answer(
    test_question,
    context
)

print("Generated Answer:\n")
print(generated_answer)

Attempt 1/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 2/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Generated Answer:

Based on the provided context, the risk factors for type 2 diabetes include:

**Primary and Demographic Factors:**
* **Age:** Risk increases with age [Page 9].
* **Overweight or obesity:** Strongly associated with a body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Page 9].
* **Prediabetes:** Having blood glucose levels that are higher than normal but not high enough to be diagnosed as diabetes [Page 5].
* **Family history:** Having a parent or sibling with diabetes [Page 9].
* **High-risk populations:** Being African America

In [37]:
# TEMPORARY: Use previously generated RAG answer
# No Gemini API call

generated_answer = """
Based on the provided medical context, the risk of type 2 diabetes increases with age and is strongly associated with overweight or obesity (BMI ≥25, or ≥23 for Asian Americans).

Other risk factors include:
- Prediabetes (glucose higher than normal but not diabetes)
- Family history of diabetes (parent or sibling)
- Being a member of a high-risk population
- History of gestational diabetes mellitus (GDM)
- Physical inactivity
- Hypertension
- HDL-C ≤35 mg/dL
- Fasting triglycerides ≥250 mg/dL
- Conditions associated with insulin resistance, such as acanthosis nigricans, NASH, and PCOS
- Atherosclerotic cardiovascular disease
- Depression
- Treatment with atypical antipsychotic medications

Emerging risk factors also include obstructive sleep apnea and chronic sleep deprivation (<6 hours/day).
"""

print("Cached test answer loaded.")
print("Answer length:", len(generated_answer))

Cached test answer loaded.
Answer length: 806


In [38]:
claims = extract_claims(generated_answer)

print("Total claims detected:", len(claims))

for i, claim in enumerate(claims, start=1):
    print(f"{i}. {claim}")

Total claims detected: 3
1. Based on the provided medical context, the risk of type 2 diabetes increases with age and is strongly associated with overweight or obesity (BMI ≥25, or ≥23 for Asian Americans).
2. Other risk factors include: Prediabetes (glucose higher than normal but not diabetes) Family history of diabetes (parent or sibling) Being a member of a high-risk population History of gestational diabetes mellitus (GDM) Physical inactivity Hypertension HDL-C ≤35 mg/dL Fasting triglycerides ≥250 mg/dL Conditions associated with insulin resistance, such as acanthosis nigricans, NASH, and PCOS Atherosclerotic cardiovascular disease Depression Treatment with atypical antipsychotic medications
3. Emerging risk factors also include obstructive sleep apnea and chronic sleep deprivation (<6 hours/day).


In [41]:
print("Answer generated:", generated_answer is not None)
print("Answer length:", len(generated_answer))

Answer generated: True
Answer length: 1420


In [9]:
# Cell 9 — Inspect Retrieval with Larger Top-K

retrieved_docs_10 = retrieve_documents(
    test_question,
    top_k=10
)

print("Retrieved documents:", len(retrieved_docs_10))

for doc_item in retrieved_docs_10:
    print(
        f"\nRank {doc_item['rank']} | "
        f"Page {doc_item['page']} | "
        f"Distance {doc_item['distance']:.4f}"
    )
    print(doc_item["text"][:500], "...")

Retrieved documents: 10

Rank 1 | Page 5 | Distance 0.6296
5 INTRODUCTION The diabetes problem Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327 billion, including $237 billion in direct medical costs and $90 billion in reduced p ...

Rank 2 | Page 5 | Distance 0.7152
diabetes have A1C > 8; 28 percent have BP > 140/90; 51 percent are on statins; 44 percent have LDL > 100; and 20 percent use tobacco. Thus, a substantial proportion of people with diabetes do not meet goals generally agreed as appropriate for most individuals with diabetes.5 The National Institutes of Health (NIH)-sponsored Diabetes Prevention Program clinical trial proved that type 2 diabetes can 

In [10]:
# Cell 10 — Generate Answer Using Top-10 Retrieved Evidence

retrieved_docs = retrieve_documents(
    test_question,
    top_k=10
)

context = build_context(retrieved_docs)

generated_answer = generate_grounded_answer(
    test_question,
    context
)

print("Generated Answer:\n")
print(generated_answer)

Attempt 1/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 2/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 3/3 failed: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}
Generated Answer:

None


In [11]:
# Cell 11 — Load Hallucination Detection Model

NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME
)

nli_model.eval()

print("NLI model loaded successfully.")
print("Number of labels:", nli_model.config.num_labels)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI model loaded successfully.
Number of labels: 3


In [12]:
# Cell 12 — Hallucination Verification Functions

NLI_LABELS = {
    0: "contradiction",
    1: "entailment",
    2: "neutral"
}


def semantic_similarity(answer, documents):
    """
    Calculate maximum semantic similarity between the answer
    and retrieved document chunks.
    """

    if not answer or not documents:
        return 0.0

    answer_embedding = embedding_model.encode(
        [answer],
        normalize_embeddings=True
    )[0]

    document_embeddings = embedding_model.encode(
        [doc["text"] for doc in documents],
        normalize_embeddings=True
    )

    similarities = np.dot(
        document_embeddings,
        answer_embedding
    )

    return float(np.max(similarities))


def nli_score(premise, hypothesis):
    """
    Calculate contradiction, entailment and neutral probabilities.
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )[0].cpu().numpy()

    return {
        "contradiction": float(probabilities[0]),
        "entailment": float(probabilities[1]),
        "neutral": float(probabilities[2])
    }


def check_nli_against_documents(answer, documents):
    """
    Find the strongest NLI relationship between the answer
    and retrieved documents.
    """

    best_result = {
        "contradiction": 0.0,
        "entailment": 0.0,
        "neutral": 0.0
    }

    for doc in documents:
        result = nli_score(
            doc["text"],
            answer
        )

        if result["entailment"] > best_result["entailment"]:
            best_result = result

    return best_result


def verify_answer(answer, documents):
    """
    Combine semantic similarity and NLI into a support score.
    """

    similarity = semantic_similarity(
        answer,
        documents
    )

    nli_result = check_nli_against_documents(
        answer,
        documents
    )

    support_score = (
        0.5 * similarity +
        0.5 * nli_result["entailment"]
    )

    if nli_result["contradiction"] >= 0.50:
        decision = "CONTRADICTED"

    elif support_score >= 0.60:
        decision = "SUPPORTED"

    else:
        decision = "POTENTIAL HALLUCINATION"

    return {
        "semantic_similarity": similarity,
        "contradiction": nli_result["contradiction"],
        "entailment": nli_result["entailment"],
        "neutral": nli_result["neutral"],
        "support_score": support_score,
        "decision": decision
    }

In [13]:
# Cell 13 — Verify Generated Answer

verification = verify_answer(
    generated_answer,
    retrieved_docs
)

print("Hallucination Verification")
print("-" * 40)

for key, value in verification.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Hallucination Verification
----------------------------------------
semantic_similarity: 0.8119
contradiction: 0.0003
entailment: 0.9849
neutral: 0.0148
support_score: 0.8984
decision: SUPPORTED


In [40]:
# Cell 14 — Robust Claim Extraction

import re


def extract_claims(answer):
    """
    Extract individual factual claims from a medical answer.

    Handles:
    - Bullet lists
    - Numbered lists
    - Newline-separated lists
    - Comma-separated risk factors
    """

    if not answer or not answer.strip():
        return []

    text = answer.strip()

    # Remove source/page citations
    text = re.sub(r"\([^)]*Page\s+\d+[^)]*\)", "", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n")

    claims = []

    # --------------------------------------------------
    # 1. Split by actual lines
    # --------------------------------------------------

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Remove bullets
        line = re.sub(r"^\s*[-*•]\s*", "", line)

        # Remove numbering
        line = re.sub(r"^\s*\d+[\.\)]\s*", "", line)

        line = line.strip()

        if not line:
            continue

        # Skip headings
        if line.lower() in {
            "other risk factors include:",
            "additional risk factors include:",
            "emerging risk factors also include:"
        }:
            continue

        claims.append(line)

    # --------------------------------------------------
    # 2. If lists were collapsed into one line,
    #    explicitly split known risk-factor patterns
    # --------------------------------------------------

    expanded_claims = []

    for claim in claims:

        lower = claim.lower()

        if "other risk factors include:" in lower:

            prefix, risk_text = re.split(
                r"other risk factors include:",
                claim,
                maxsplit=1,
                flags=re.IGNORECASE
            )

            if prefix.strip():
                expanded_claims.append(prefix.strip())

            risk_text = risk_text.strip()

            # Known risk-factor boundaries
            patterns = [
                r"(?=Family history of diabetes)",
                r"(?=Being a member of a high-risk population)",
                r"(?=History of gestational diabetes)",
                r"(?=Physical inactivity)",
                r"(?=Hypertension)",
                r"(?=HDL-C)",
                r"(?=Fasting triglycerides)",
                r"(?=Conditions associated with insulin resistance)",
                r"(?=Atherosclerotic cardiovascular disease)",
                r"(?=Depression)",
                r"(?=Treatment with atypical antipsychotic medications)"
            ]

            parts = re.split("|".join(patterns), risk_text)

            for part in parts:

                part = part.strip()

                if not part:
                    continue

                # Remove accidental punctuation
                part = part.rstrip(".,;")

                expanded_claims.append(part)

        else:
            expanded_claims.append(claim)

    # --------------------------------------------------
    # 3. Split comma-separated first risk statement only
    # --------------------------------------------------

    final_claims = []

    for claim in expanded_claims:

        # Don't split normal sentences by commas.
        # Only handle the specific BMI/risk sentence as one claim.
        final_claims.append(claim.strip())

    # --------------------------------------------------
    # 4. Remove duplicates / empty claims
    # --------------------------------------------------

    unique_claims = []
    seen = set()

    for claim in final_claims:

        claim = re.sub(r"\s+", " ", claim).strip()

        if len(claim) < 5:
            continue

        normalized = claim.lower()

        if normalized not in seen:
            seen.add(normalized)
            unique_claims.append(claim)

    return unique_claims


# Test
claims = extract_claims(generated_answer)

print("Total claims detected:", len(claims))
print("-" * 70)

for i, claim in enumerate(claims, start=1):
    print(f"{i}. {claim}")

Total claims detected: 20
----------------------------------------------------------------------
1. Based on the provided context, the risk factors for type 2 diabetes include:
2. *Primary and Demographic Factors:**
3. **Age:** Risk increases with age [Page 9].
4. **Overweight or obesity:** Strongly associated with a body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Page 9].
5. **Prediabetes:** Having blood glucose levels that are higher than normal but not high enough to be diagnosed as diabetes [Page 5].
6. **Family history:** Having a parent or sibling with diabetes [Page 9].
7. **High-risk populations:** Being African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, or Pacific Islander American [Page 9].
8. *Medical History and Associated Conditions:**
9. History of gestational diabetes mellitus (GDM) [Page 9].
10. Physical inactivity [Page 9].
11. Hypertension [Page 9].
12. High-density lipoprotein cholesterol (HDL-C) level ≤ 35 mg/dL 

In [42]:
# Cell 14 — Final Claim Extraction

import re

def extract_claims(answer):

    if not answer or not answer.strip():
        return []

    text = answer.strip()

    # Remove page citations
    text = re.sub(r"\s*\[Page\s+\d+\]", "", text)

    # Remove markdown bold/italic markers
    text = re.sub(r"\*\*", "", text)
    text = re.sub(r"(?<!\*)\*(?!\*)", "", text)

    lines = text.splitlines()

    claims = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Remove bullets / numbering
        line = re.sub(r"^\s*[-•*]\s*", "", line)
        line = re.sub(r"^\s*\d+[\.\)]\s*", "", line)

        line = line.strip()

        if not line:
            continue

        lower = line.lower()

        # Ignore headings
        headings = [
            "primary and demographic factors:",
            "medical history and associated conditions:",
            "emerging risk factors:",
            "other risk factors include:",
            "additional risk factors include:"
        ]

        if any(lower.startswith(h) for h in headings):
            continue

        # Ignore umbrella introduction
        if lower.startswith(
            "based on the provided context, the risk factors"
        ):
            continue

        # Ignore incomplete generated claims
        if "[medications]" in lower:
            line = line.replace("[medications]", "medications")

        # Ignore obvious incomplete fragment
        if lower.endswith("atypical"):
            continue

        claims.append(line)

    # Remove duplicates
    unique_claims = []
    seen = set()

    for claim in claims:

        claim = re.sub(r"\s+", " ", claim).strip()
        normalized = claim.lower()

        if normalized not in seen:
            seen.add(normalized)
            unique_claims.append(claim)

    return unique_claims


claims = extract_claims(generated_answer)

print("Total factual claims:", len(claims))
print("-" * 70)

for i, claim in enumerate(claims, start=1):
    print(f"{i}. {claim}")

Total factual claims: 16
----------------------------------------------------------------------
1. Age: Risk increases with age.
2. Overweight or obesity: Strongly associated with a body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans).
3. Prediabetes: Having blood glucose levels that are higher than normal but not high enough to be diagnosed as diabetes.
4. Family history: Having a parent or sibling with diabetes.
5. High-risk populations: Being African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, or Pacific Islander American.
6. History of gestational diabetes mellitus (GDM).
7. Physical inactivity.
8. Hypertension.
9. High-density lipoprotein cholesterol (HDL-C) level ≤ 35 mg/dL (0.90 mmol/L).
10. Fasting triglyceride (TG) level ≥ 250 mg/dL (2.82 mmol/L).
11. Acanthosis nigricans, nonalcoholic steatohepatitis, polycystic ovary syndrome, and other conditions associated with insulin resistance.
12. Atherosclerotic cardiovascular disease.
13

In [15]:
# Cell 15 — Claim-Level Evidence Verification

def normalize_claim(claim):
    """Clean a claim before evidence matching."""
    claim = re.sub(r"\s+", " ", claim).strip()
    claim = re.sub(r"\[[^\]]*\]", "", claim).strip()
    return claim


def find_best_evidence(claim, documents, top_n=5):
    """
    Find the strongest supporting evidence for a claim
    using semantic similarity + NLI.
    """

    claim = normalize_claim(claim)

    claim_embedding = embedding_model.encode(
        [claim],
        normalize_embeddings=True
    )[0]

    document_embeddings = embedding_model.encode(
        [doc["text"] for doc in documents],
        normalize_embeddings=True
    )

    similarities = np.dot(
        document_embeddings,
        claim_embedding
    )

    # Select top semantic candidates
    candidate_indices = np.argsort(similarities)[::-1][:top_n]

    best_evidence = None

    for idx in candidate_indices:
        doc = documents[idx]

        nli_result = nli_score(
            doc["text"],
            claim
        )

        evidence_score = (
            0.50 * float(similarities[idx])
            + 0.50 * nli_result["entailment"]
            - 0.20 * nli_result["contradiction"]
        )

        result = {
            "chunk_id": doc["chunk_id"],
            "page": doc["page"],
            "text": doc["text"],
            "similarity": float(similarities[idx]),
            "entailment": nli_result["entailment"],
            "contradiction": nli_result["contradiction"],
            "neutral": nli_result["neutral"],
            "evidence_score": evidence_score
        }

        if (
            best_evidence is None
            or result["evidence_score"]
            > best_evidence["evidence_score"]
        ):
            best_evidence = result

    return best_evidence


# Test on all extracted claims
claim_results = []

for i, claim in enumerate(claims, start=1):
    evidence = find_best_evidence(
        claim,
        retrieved_docs,
        top_n=5
    )

    claim_results.append({
        "claim_number": i,
        "claim": claim,
        "evidence": evidence
    })

print("Claims verified:", len(claim_results))

for item in claim_results:
    evidence = item["evidence"]

    print(
        f"\nClaim {item['claim_number']}: "
        f"{item['claim'][:100]}"
    )

    if evidence:
        print(
            f"Best evidence → Page {evidence['page']} | "
            f"Similarity: {evidence['similarity']:.4f} | "
            f"Entailment: {evidence['entailment']:.4f} | "
            f"Contradiction: {evidence['contradiction']:.4f} | "
            f"Score: {evidence['evidence_score']:.4f}"
        )

Claims verified: 0


In [44]:
# Improved claim -> verification hypothesis

def claim_to_hypothesis(claim):

    claim = normalize_claim(claim)
    lower = claim.lower()

    # Remove labels before verification
    claim_clean = re.sub(
        r"^(age|overweight or obesity|prediabetes|family history|"
        r"high-risk populations|history of gestational diabetes mellitus \(gdm\)|"
        r"physical inactivity|hypertension|"
        r"high-density lipoprotein cholesterol \(hdl-c\) level|"
        r"fasting triglyceride \(tg\) level):\s*",
        "",
        claim,
        flags=re.IGNORECASE
    ).strip()

    lower_clean = claim_clean.lower()

    # Age
    if lower.startswith("age:"):
        return "Increasing age is a risk factor for type 2 diabetes."

    # Obesity
    if lower.startswith("overweight or obesity:"):
        return (
            "Overweight or obesity is a risk factor for type 2 diabetes, "
            "with increased risk associated with BMI ≥25 kg/m² "
            "or ≥23 kg/m² for Asian Americans."
        )

    # Prediabetes
    if lower.startswith("prediabetes:"):
        return (
            "Prediabetes is a risk factor for type 2 diabetes."
        )

    # Family history
    if lower.startswith("family history:"):
        return (
            "Having a parent or sibling with diabetes "
            "is a risk factor for type 2 diabetes."
        )

    # High-risk populations
    if lower.startswith("high-risk populations:"):
        return (
            "Being African American, Hispanic or Latino, "
            "American Indian, Alaska Native, Asian American, "
            "or Pacific Islander American is a risk factor "
            "for type 2 diabetes."
        )

    # Gestational diabetes
    if "gestational diabetes" in lower:
        return (
            "A history of gestational diabetes mellitus "
            "is a risk factor for type 2 diabetes."
        )

    # Physical inactivity
    if "physical inactivity" in lower:
        return (
            "Physical inactivity is a risk factor for type 2 diabetes."
        )

    # Hypertension
    if lower.startswith("hypertension"):
        return (
            "Hypertension is a risk factor for type 2 diabetes."
        )

    # HDL
    if "hdl-c" in lower or "high-density lipoprotein" in lower:
        return (
            "A high-density lipoprotein cholesterol level "
            "of 35 mg/dL or lower is a risk factor for type 2 diabetes."
        )

    # Triglycerides
    if "triglyceride" in lower:
        return (
            "A fasting triglyceride level of 250 mg/dL or higher "
            "is a risk factor for type 2 diabetes."
        )

    # Insulin resistance
    if (
        "acanthosis nigricans" in lower
        or "insulin resistance" in lower
        or "polycystic ovary syndrome" in lower
        or "nonalcoholic steatohepatitis" in lower
    ):
        return (
            "Acanthosis nigricans, nonalcoholic steatohepatitis, "
            "polycystic ovary syndrome, and other conditions "
            "associated with insulin resistance are risk factors "
            "for type 2 diabetes."
        )

    # Cardiovascular disease
    if "atherosclerotic cardiovascular disease" in lower:
        return (
            "Atherosclerotic cardiovascular disease is a risk factor "
            "for type 2 diabetes."
        )

    # Depression
    if lower.startswith("depression"):
        return (
            "Depression is a risk factor for type 2 diabetes."
        )

    # Incomplete claim
    if "atypical medications" in lower:
        return claim

    # Sleep apnea
    if "obstructive sleep apnea" in lower:
        return (
            "Obstructive sleep apnea is an emerging risk factor "
            "for type 2 diabetes."
        )

    # Sleep deprivation
    if "sleep deprivation" in lower:
        return (
            "Chronic sleep deprivation of less than 6 hours per day "
            "is an emerging risk factor for type 2 diabetes."
        )

    return claim

In [45]:
for i, claim in enumerate(claims, start=1):
    print(f"{i}. CLAIM:")
    print(claim)
    print("   HYPOTHESIS:")
    print(claim_to_hypothesis(claim))
    print("-" * 70)

1. CLAIM:
Age: Risk increases with age.
   HYPOTHESIS:
Increasing age is a risk factor for type 2 diabetes.
----------------------------------------------------------------------
2. CLAIM:
Overweight or obesity: Strongly associated with a body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans).
   HYPOTHESIS:
Overweight or obesity is a risk factor for type 2 diabetes, with increased risk associated with BMI ≥25 kg/m² or ≥23 kg/m² for Asian Americans.
----------------------------------------------------------------------
3. CLAIM:
Prediabetes: Having blood glucose levels that are higher than normal but not high enough to be diagnosed as diabetes.
   HYPOTHESIS:
Prediabetes is a risk factor for type 2 diabetes.
----------------------------------------------------------------------
4. CLAIM:
Family history: Having a parent or sibling with diabetes.
   HYPOTHESIS:
Having a parent or sibling with diabetes is a risk factor for type 2 diabetes.
-----------------------------------

In [29]:
# Cell 17 — Claim-Specific Retrieval

def retrieve_for_claim(claim, top_k=5):
    """
    Retrieve the most relevant medical chunks specifically
    for an individual claim.
    """

    hypothesis = claim_to_hypothesis(claim)

    query_embedding = embedding_model.encode(
        [hypothesis]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "source": (
                "NIDDK Guiding Principles for the Care "
                "of People with or at Risk for Diabetes"
            ),
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distances[0][rank - 1])
        })

    return results


# Test with a direct claim instead of claims[2]
test_claim = "Family history of diabetes (parent or sibling)"

claim_docs = retrieve_for_claim(
    test_claim,
    top_k=5
)

print("Claim:")
print(test_claim)

print("\nRetrieved evidence:")

for doc_item in claim_docs:
    print(
        f"\nRank {doc_item['rank']} | "
        f"Page {doc_item['page']} | "
        f"Distance {doc_item['distance']:.4f}"
    )
    print(doc_item["text"][:500], "...")

Claim:
Family history of diabetes (parent or sibling)

Retrieved evidence:

Rank 1 | Page 74 | Distance 0.7931
74 Children and adolescents Diabetes is one of the most common chronic conditions in school-age children in the United States. About 193,000 youth under 20 years old have diabetes, 0.24 percent of all in this age group.1 The incidence of both type 1 and type 2 diabetes in youth is increasing.2 Type 1 diabetes accounts for nearly all diabetes in children under age 10. After age 10, type 1 is the most common form in U.S. youth overall, but type 2 is more common in new cases among minority groups,  ...

Rank 2 | Page 9 | Distance 0.8092
and follow-up may alert people to the onset of type 1 diabetes and lower risk for DKA. However, the population impact of such screening is limited by the fact that only 10 percent of people with type 1 diabetes have a family history of the disease.4 Table 1. Risk Factors for Type 2 Diabetes ...

Rank 3 | Page 6 | Distance 0.8195
general agreement 

In [31]:
# Cell 17 — Claim-Specific Retrieval

def retrieve_for_claim(claim, top_k=5):
    """
    Retrieve the most relevant medical chunks specifically
    for an individual claim.
    """

    hypothesis = claim_to_hypothesis(claim)

    query_embedding = embedding_model.encode(
        [hypothesis]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "source": (
                "NIDDK Guiding Principles for the Care "
                "of People with or at Risk for Diabetes"
            ),
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distances[0][rank - 1])
        })

    return results


# Test with a direct claim instead of claims[2]
test_claim = "Family history of diabetes (parent or sibling)"

claim_docs = retrieve_for_claim(
    test_claim,
    top_k=5
)

print("Claim:")
print(test_claim)

print("\nRetrieved evidence:")

for doc_item in claim_docs:
    print(
        f"\nRank {doc_item['rank']} | "
        f"Page {doc_item['page']} | "
        f"Distance {doc_item['distance']:.4f}"
    )
    print(doc_item["text"][:500], "...")

Claim:
Family history of diabetes (parent or sibling)

Retrieved evidence:

Rank 1 | Page 74 | Distance 0.7931
74 Children and adolescents Diabetes is one of the most common chronic conditions in school-age children in the United States. About 193,000 youth under 20 years old have diabetes, 0.24 percent of all in this age group.1 The incidence of both type 1 and type 2 diabetes in youth is increasing.2 Type 1 diabetes accounts for nearly all diabetes in children under age 10. After age 10, type 1 is the most common form in U.S. youth overall, but type 2 is more common in new cases among minority groups,  ...

Rank 2 | Page 9 | Distance 0.8092
and follow-up may alert people to the onset of type 1 diabetes and lower risk for DKA. However, the population impact of such screening is limited by the fact that only 10 percent of people with type 1 diabetes have a family history of the disease.4 Table 1. Risk Factors for Type 2 Diabetes ...

Rank 3 | Page 6 | Distance 0.8195
general agreement 

In [50]:
# Cell 18 — Claim Support Decision

def is_claim_supported(evidence):

    if evidence is None:
        return False

    entailment = evidence["entailment"]
    contradiction = evidence["contradiction"]
    similarity = evidence["similarity"]
    evidence_score = evidence["evidence_score"]

    # Strong direct evidence
    if (
        entailment >= 0.80
        and contradiction < 0.20
        and evidence_score >= 0.65
    ):
        return True

    # Good combined semantic + NLI evidence
    if (
        entailment >= 0.65
        and similarity >= 0.45
        and contradiction < 0.10
        and evidence_score >= 0.55
    ):
        return True

    return False

In [51]:
# Cell 19 — Optimized Claim-Level Verification
# NLI-first evidence scoring

def verify_claims_fast(answer, top_n=10, batch_size=16):

    # --------------------------------------------------
    # 1. Extract claims
    # --------------------------------------------------

    all_claims = extract_claims(answer)

    # Remove incomplete/truncated claims
    claims = [
        claim for claim in all_claims
        if "context cuts off" not in claim.lower()
        and not claim.lower().endswith("atypical")
    ]

    print(f"Total extracted claims: {len(all_claims)}")
    print(f"Valid claims for verification: {len(claims)}")
    print("-" * 70)

    if not claims:
        return {
            "claims": [],
            "supported_claims": 0,
            "total_claims": 0,
            "consistency_score": 0.0
        }

    # --------------------------------------------------
    # 2. Convert claims to verification hypotheses
    # --------------------------------------------------

    hypotheses = [
        claim_to_hypothesis(claim)
        for claim in claims
    ]

    # --------------------------------------------------
    # 3. Encode hypotheses
    # --------------------------------------------------

    print("Encoding claim hypotheses...")

    hypothesis_embeddings = embedding_model.encode(
        hypotheses,
        normalize_embeddings=True
    )

    raw_query_embeddings = embedding_model.encode(
        hypotheses
    ).astype("float32")

    # --------------------------------------------------
    # 4. Retrieve evidence candidates
    # --------------------------------------------------

    search_k = min(top_n, len(chunks))

    distances, indices = index.search(
        raw_query_embeddings,
        search_k
    )

    # --------------------------------------------------
    # 5. Normalize document embeddings
    # --------------------------------------------------

    normalized_embeddings = embeddings / (
        np.linalg.norm(
            embeddings,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    # --------------------------------------------------
    # 6. Build NLI pairs
    # --------------------------------------------------

    premises = []
    nli_hypotheses = []
    metadata = []

    for claim_idx in range(len(claims)):

        for rank, chunk_idx in enumerate(
            indices[claim_idx]
        ):

            chunk_idx = int(chunk_idx)

            similarity = float(
                np.dot(
                    normalized_embeddings[chunk_idx],
                    hypothesis_embeddings[claim_idx]
                )
            )

            premises.append(
                chunks[chunk_idx]["text"]
            )

            nli_hypotheses.append(
                hypotheses[claim_idx]
            )

            metadata.append({
                "claim_idx": claim_idx,
                "rank": rank + 1,
                "chunk": chunks[chunk_idx],
                "similarity": similarity
            })

    print(f"NLI pairs: {len(premises)}")
    print("Running batched NLI...")

    # --------------------------------------------------
    # 7. Batched NLI
    # --------------------------------------------------

    nli_results = nli_score_batch(
        premises,
        nli_hypotheses,
        batch_size=batch_size
    )

    # --------------------------------------------------
    # 8. Select strongest evidence
    # --------------------------------------------------

    best_evidence = [None] * len(claims)

    for meta, nli in zip(
        metadata,
        nli_results
    ):

        # NLI-first scoring
        evidence_score = (
            0.70 * nli["entailment"]
            + 0.30 * meta["similarity"]
            - 0.30 * nli["contradiction"]
        )

        evidence = {
            "chunk_id": meta["chunk"]["chunk_id"],
            "page": meta["chunk"]["page"],
            "text": meta["chunk"]["text"],
            "similarity": meta["similarity"],
            "entailment": nli["entailment"],
            "contradiction": nli["contradiction"],
            "neutral": nli["neutral"],
            "evidence_score": evidence_score
        }

        claim_idx = meta["claim_idx"]

        if (
            best_evidence[claim_idx] is None
            or evidence_score
            > best_evidence[claim_idx]["evidence_score"]
        ):
            best_evidence[claim_idx] = evidence

    # --------------------------------------------------
    # 9. Evaluate claim support
    # --------------------------------------------------

    results = []
    supported_count = 0

    for i, (claim, evidence) in enumerate(
        zip(claims, best_evidence),
        start=1
    ):

        supported = is_claim_supported(evidence)

        if supported:
            supported_count += 1

        status = (
            "SUPPORTED"
            if supported
            else "NOT SUPPORTED"
        )

        print(
            f"Claim {i}: {status} | "
            f"Page {evidence['page']} | "
            f"Score {evidence['evidence_score']:.4f}"
        )

        results.append({
            "claim_number": i,
            "claim": claim,
            "hypothesis": claim_to_hypothesis(claim),
            "supported": supported,
            "evidence": evidence
        })

    # --------------------------------------------------
    # 10. Calculate consistency
    # --------------------------------------------------

    consistency_score = (
        supported_count / len(claims)
    )

    print("-" * 70)

    print(
        f"Supported: "
        f"{supported_count}/{len(claims)}"
    )

    print(
        f"Consistency Score: "
        f"{consistency_score:.4f}"
    )

    return {
        "claims": results,
        "supported_claims": supported_count,
        "total_claims": len(claims),
        "consistency_score": consistency_score
    }


# ------------------------------------------------------
# Run claim verification
# ------------------------------------------------------

claim_verification = verify_claims_fast(
    generated_answer,
    top_n=10,
    batch_size=16
)

Total extracted claims: 16
Valid claims for verification: 15
----------------------------------------------------------------------
Encoding claim hypotheses...
NLI pairs: 150
Running batched NLI...
Claim 1: NOT SUPPORTED | Page 26 | Score 0.2188
Claim 2: SUPPORTED | Page 9 | Score 0.8879
Claim 3: SUPPORTED | Page 76 | Score 0.8861
Claim 4: NOT SUPPORTED | Page 9 | Score 0.1929
Claim 5: SUPPORTED | Page 9 | Score 0.8898
Claim 6: SUPPORTED | Page 76 | Score 0.9091
Claim 7: SUPPORTED | Page 37 | Score 0.7925
Claim 8: NOT SUPPORTED | Page 63 | Score 0.2209
Claim 9: SUPPORTED | Page 9 | Score 0.8565
Claim 10: SUPPORTED | Page 9 | Score 0.8357
Claim 11: SUPPORTED | Page 9 | Score 0.8527
Claim 12: SUPPORTED | Page 9 | Score 0.7941
Claim 13: SUPPORTED | Page 9 | Score 0.8254
Claim 14: SUPPORTED | Page 9 | Score 0.8962
Claim 15: NOT SUPPORTED | Page 9 | Score 0.5802
----------------------------------------------------------------------
Supported: 11/15
Consistency Score: 0.7333


In [35]:
# Cell 19A — Check Current Pipeline State

print("generated_answer type:", type(generated_answer))
print("generated_answer is None:", generated_answer is None)

if generated_answer:
    print("\nGenerated answer:")
    print(generated_answer)

    test_claims = extract_claims(generated_answer)

    print("\nClaims detected:", len(test_claims))

    for i, claim in enumerate(test_claims, start=1):
        print(f"{i}. {claim}")
else:
    print("\nNo generated answer is currently available.")

generated_answer type: <class 'NoneType'>
generated_answer is None: True

No generated answer is currently available.


In [48]:
# Cell 19B — Inspect Unsupported Claims

print("UNSUPPORTED CLAIMS")
print("=" * 80)

for result in claim_verification["claims"]:

    if not result["supported"]:

        evidence = result["evidence"]

        print("\nClaim:")
        print(result["claim"])

        print("\nHypothesis:")
        print(result["hypothesis"])

        print("\nSelected Evidence:")
        print(
            f"Page: {evidence['page']}\n"
            f"Chunk: {evidence['chunk_id']}\n"
            f"Similarity: {evidence['similarity']:.4f}\n"
            f"Entailment: {evidence['entailment']:.4f}\n"
            f"Contradiction: {evidence['contradiction']:.4f}\n"
            f"Neutral: {evidence['neutral']:.4f}\n"
            f"Evidence Score: {evidence['evidence_score']:.4f}"
        )

        print("\nEvidence Text:")
        print(evidence["text"][:1000])

        print("\n" + "-" * 80)

UNSUPPORTED CLAIMS

Claim:
Age: Risk increases with age.

Hypothesis:
Increasing age is a risk factor for type 2 diabetes.

Selected Evidence:
Page: 26
Chunk: page_26_chunk_0
Similarity: 0.6327
Entailment: 0.0417
Contradiction: 0.0005
Neutral: 0.9578
Evidence Score: 0.3371

Evidence Text:
26 GUIDING PRINCIPLES FOR THE CARE OF PEOPLE WITH OR AT RISK FOR DIABETES | PRINCIPLE 3 • American Diabetes Association, European Association for the Study of Diabetes. Management of Hyperglycemia in Type 2 Diabetes: A Patient-Centered Approach: Position Statement of the American Diabetes Association (ADA) and the European Association for the Study of Diabetes (EASD). 2012. • American Geriatrics Society. Guidelines for Improving the Care of Older Adults with Diabetes Mellitus: 2013 Update and Supplemental Information. 2013. • VA/DoD Clinical Practice Guidelines. Management of Type 2 Diabetes Mellitus in Primary Care – 2017.

-----------------------------------------------------------------------------

In [49]:
# Cell 19C — Targeted Evidence Verification Test

test_claims = [
    claims[0],   # Age
    claims[3],   # Family history
    claims[7],   # Hypertension
    claims[15-1] # Chronic sleep deprivation
]

for claim in test_claims:

    hypothesis = claim_to_hypothesis(claim)

    print("\n" + "=" * 80)
    print("CLAIM:")
    print(claim)

    print("\nHYPOTHESIS:")
    print(hypothesis)

    evidence = find_best_evidence(
        claim,
        top_n=30
    )

    print("\nBEST EVIDENCE:")
    print("Page:", evidence["page"])
    print("Similarity:", round(evidence["similarity"], 4))
    print("Entailment:", round(evidence["entailment"], 4))
    print("Contradiction:", round(evidence["contradiction"], 4))
    print("Evidence Score:", round(evidence["evidence_score"], 4))

    print("\nTEXT:")
    print(evidence["text"][:1200])


CLAIM:
Age: Risk increases with age.

HYPOTHESIS:
Increasing age is a risk factor for type 2 diabetes.

BEST EVIDENCE:
Page: 9
Similarity: 0.544
Entailment: 0.9904
Contradiction: 0.0002
Evidence Score: 0.7672

TEXT:
9 Adapted from American Diabetes Association Standards of Care in Diabetes—2018 Risk of type 2 diabetes increases with age and is strongly associated with overweight or obesity—body mass index (BMI) ≥ 25 kg/m2 (≥ 23 kg/m2 for Asian Americans5) Additional risk factors include 1. 2. 3. 4. 5. Family history of diabetes (i.e., parent or sibling) Member of high-risk population: African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, Pacific Islander American History of GDM Physical inactivity Hypertension Obstructive sleep apnea and chronic sleep deprivation (< 6 hours/day) are emerging risk factors. 6. 7. 8. 9. 10. 11. High-density lipoprotein cholesterol (HDL-C) level ≤ 35 mg/dL (0.90 mmol/L) Fasting triglyceride (TG) level ≥ 250 mg/dL (2.82 mmol/L)

In [22]:
print("generated_answer:", generated_answer)

generated_answer: None


In [53]:
# Cell 20 — Final Confidence Scoring Integration

def calculate_final_confidence(
    answer,
    question,
    retrieved_docs,
    claim_verification
):
    """
    Calculate final confidence using:

    30% Answer Semantic Similarity
    30% Claim-level NLI Entailment
    20% Retrieval Quality
    20% Claim Consistency
    """

    # --------------------------------------------------
    # 1. Answer Semantic Similarity
    # --------------------------------------------------

    if not answer or not retrieved_docs:
        semantic_similarity = 0.0

    else:

        answer_embedding = embedding_model.encode(
            [answer],
            normalize_embeddings=True
        )[0]

        document_embeddings = embedding_model.encode(
            [doc["text"] for doc in retrieved_docs],
            normalize_embeddings=True
        )

        answer_similarities = np.dot(
            document_embeddings,
            answer_embedding
        )

        semantic_similarity = float(
            np.max(answer_similarities)
        )

        semantic_similarity = max(
            0.0,
            min(1.0, semantic_similarity)
        )

    # --------------------------------------------------
    # 2. Claim-level NLI Entailment
    # --------------------------------------------------

    valid_claim_results = [
        result
        for result in claim_verification["claims"]
        if result["supported"] is not None
    ]

    if valid_claim_results:

        entailment_scores = [
            result["evidence"]["entailment"]
            for result in valid_claim_results
        ]

        nli_entailment = float(
            np.mean(entailment_scores)
        )

    else:
        nli_entailment = 0.0

    nli_entailment = max(
        0.0,
        min(1.0, nli_entailment)
    )

    # --------------------------------------------------
    # 3. Retrieval Quality
    # --------------------------------------------------

    if retrieved_docs:

        query_embedding = embedding_model.encode(
            [question],
            normalize_embeddings=True
        )[0]

        retrieval_embeddings = embedding_model.encode(
            [doc["text"] for doc in retrieved_docs],
            normalize_embeddings=True
        )

        retrieval_similarities = np.dot(
            retrieval_embeddings,
            query_embedding
        )

        # Use top evidence rather than weak average
        top_k = min(5, len(retrieval_similarities))

        top_scores = np.sort(
            retrieval_similarities
        )[-top_k:]

        retrieval_quality = float(
            np.mean(top_scores)
        )

    else:
        retrieval_quality = 0.0

    retrieval_quality = max(
        0.0,
        min(1.0, retrieval_quality)
    )

    # --------------------------------------------------
    # 4. Claim Consistency
    # --------------------------------------------------

    consistency_score = float(
        claim_verification["consistency_score"]
    )

    # --------------------------------------------------
    # 5. Final Confidence
    # --------------------------------------------------

    confidence_score = (
        0.30 * semantic_similarity
        + 0.30 * nli_entailment
        + 0.20 * retrieval_quality
        + 0.20 * consistency_score
    )

    confidence_score = max(
        0.0,
        min(1.0, confidence_score)
    )

    # --------------------------------------------------
    # 6. Confidence Level
    # --------------------------------------------------

    if confidence_score >= 0.80:
        confidence_level = "HIGH"

    elif confidence_score >= 0.60:
        confidence_level = "MEDIUM"

    else:
        confidence_level = "LOW"

    # --------------------------------------------------
    # 7. Display
    # --------------------------------------------------

    print("=" * 70)
    print("FINAL CONFIDENCE SCORE")
    print("=" * 70)

    print(
        f"Answer Semantic Similarity : "
        f"{semantic_similarity:.4f}"
    )

    print(
        f"Claim NLI Entailment      : "
        f"{nli_entailment:.4f}"
    )

    print(
        f"Retrieval Quality         : "
        f"{retrieval_quality:.4f}"
    )

    print(
        f"Claim Consistency         : "
        f"{consistency_score:.4f}"
    )

    print("-" * 70)

    print(
        f"Final Confidence          : "
        f"{confidence_score:.4f}"
    )

    print(
        f"Confidence Level          : "
        f"{confidence_level}"
    )

    print("=" * 70)

    return {
        "semantic_similarity": semantic_similarity,
        "nli_entailment": nli_entailment,
        "retrieval_quality": retrieval_quality,
        "claim_consistency": consistency_score,
        "confidence_score": confidence_score,
        "confidence_level": confidence_level
    }


# ------------------------------------------------------
# Run final confidence calculation
# ------------------------------------------------------

confidence_result = calculate_final_confidence(
    answer=generated_answer,
    question=test_question,
    retrieved_docs=retrieved_docs,
    claim_verification=claim_verification
)

FINAL CONFIDENCE SCORE
Answer Semantic Similarity : 0.7735
Claim NLI Entailment      : 0.7511
Retrieval Quality         : 0.6444
Claim Consistency         : 0.7333
----------------------------------------------------------------------
Final Confidence          : 0.7329
Confidence Level          : MEDIUM


In [54]:
# Cell 21 — Final Hallucination Decision

def determine_final_decision(
    claim_verification,
    confidence_result
):
    """
    Determine the final hallucination status.

    Priority:
    1. Strong contradiction
    2. Low claim consistency / confidence
    3. Supported
    """

    consistency = claim_verification[
        "consistency_score"
    ]

    confidence = confidence_result[
        "confidence_score"
    ]

    # --------------------------------------------------
    # Check for strong contradictions
    # --------------------------------------------------

    strong_contradictions = 0

    for result in claim_verification["claims"]:

        evidence = result["evidence"]

        if evidence["contradiction"] >= 0.50:
            strong_contradictions += 1

    # --------------------------------------------------
    # Final decision
    # --------------------------------------------------

    if strong_contradictions > 0:

        decision = "CONTRADICTED"

    elif (
        consistency < 0.60
        or confidence < 0.60
    ):

        decision = "POTENTIAL HALLUCINATION"

    else:

        decision = "SUPPORTED"

    # --------------------------------------------------
    # Display
    # --------------------------------------------------

    print("=" * 70)
    print("FINAL HALLUCINATION DECISION")
    print("=" * 70)

    print(
        f"Claim Consistency : {consistency:.4f}"
    )

    print(
        f"Confidence Score  : {confidence:.4f}"
    )

    print(
        f"Strong Contradictions : "
        f"{strong_contradictions}"
    )

    print("-" * 70)

    print(
        f"FINAL DECISION: {decision}"
    )

    print("=" * 70)

    return {
        "decision": decision,
        "claim_consistency": consistency,
        "confidence_score": confidence,
        "strong_contradictions": strong_contradictions
    }


# ------------------------------------------------------
# Run final decision
# ------------------------------------------------------

final_decision = determine_final_decision(
    claim_verification=claim_verification,
    confidence_result=confidence_result
)

FINAL HALLUCINATION DECISION
Claim Consistency : 0.7333
Confidence Score  : 0.7329
Strong Contradictions : 0
----------------------------------------------------------------------
FINAL DECISION: SUPPORTED


In [55]:
# Cell 22 — Self-Correction Integration

def should_self_correct(final_decision):
    """
    Decide whether the generated answer needs self-correction.
    """

    decision = final_decision["decision"]
    confidence = final_decision["confidence_score"]

    if decision in {
        "POTENTIAL HALLUCINATION",
        "CONTRADICTED"
    }:
        return True

    if confidence < 0.60:
        return True

    return False


def build_correction_prompt(
    question,
    answer,
    claim_verification
):
    """
    Build a grounded correction prompt using
    only the retrieved evidence associated
    with unsupported claims.
    """

    unsupported_claims = []

    for result in claim_verification["claims"]:

        if not result["supported"]:

            evidence = result["evidence"]

            unsupported_claims.append(
                {
                    "claim": result["claim"],
                    "evidence": evidence["text"],
                    "page": evidence["page"]
                }
            )

    evidence_text = ""

    for item in unsupported_claims:

        evidence_text += (
            f"\nClaim: {item['claim']}\n"
            f"Evidence (Page {item['page']}): "
            f"{item['evidence']}\n"
        )

    prompt = f"""
You are correcting a medical RAG answer.

USER QUESTION:
{question}

ORIGINAL ANSWER:
{answer}

The following claims were not sufficiently supported
during evidence verification:

{evidence_text}

Instructions:

1. Use ONLY the provided evidence.
2. Remove unsupported claims.
3. Do not invent missing information.
4. Preserve claims that are supported.
5. If evidence is insufficient, explicitly say so.
6. Keep the corrected answer concise.
7. Include source page references where appropriate.
8. Do not provide personalized medical advice.

Return ONLY the corrected answer.
"""

    return prompt


def self_correct_answer(
    question,
    answer,
    final_decision,
    claim_verification
):
    """
    Perform self-correction only when required.

    NOTE:
    The Gemini API call is separated from the decision logic
    so the pipeline can be tested even when API quota is unavailable.
    """

    correction_needed = should_self_correct(
        final_decision
    )

    print("=" * 70)
    print("SELF-CORRECTION CHECK")
    print("=" * 70)

    print(
        f"Initial Decision : "
        f"{final_decision['decision']}"
    )

    print(
        f"Initial Confidence : "
        f"{final_decision['confidence_score']:.4f}"
    )

    print(
        f"Correction Required : "
        f"{correction_needed}"
    )

    print("=" * 70)

    if not correction_needed:

        return {
            "correction_needed": False,
            "original_answer": answer,
            "corrected_answer": answer,
            "correction_applied": False
        }

    correction_prompt = build_correction_prompt(
        question,
        answer,
        claim_verification
    )

    return {
        "correction_needed": True,
        "original_answer": answer,
        "corrected_answer": None,
        "correction_applied": False,
        "correction_prompt": correction_prompt
    }


# ------------------------------------------------------
# Run self-correction check
# ------------------------------------------------------

self_correction_result = self_correct_answer(
    question=test_question,
    answer=generated_answer,
    final_decision=final_decision,
    claim_verification=claim_verification
)

SELF-CORRECTION CHECK
Initial Decision : SUPPORTED
Initial Confidence : 0.7329
Correction Required : False


In [56]:
# Cell 23 — Complete Medical RAG + Hallucination Detection Pipeline

def run_medical_rag_pipeline(
    question,
    top_k=10,
    verification_top_n=10,
    run_self_correction=True
):
    """
    Complete end-to-end medical RAG pipeline.

    Flow:
    Question
        ↓
    FAISS Retrieval
        ↓
    Grounded Gemini Answer
        ↓
    Claim Extraction
        ↓
    Claim-Level Evidence Verification
        ↓
    Confidence Scoring
        ↓
    Hallucination Decision
        ↓
    Self-Correction (if required)
    """

    print("=" * 80)
    print("MEDICAL RAG + HALLUCINATION DETECTION PIPELINE")
    print("=" * 80)

    print(f"\nQuestion: {question}")

    # --------------------------------------------------
    # 1. RETRIEVAL
    # --------------------------------------------------

    print("\n[1/6] Retrieving evidence...")

    retrieved_docs = retrieve_documents(
        question,
        top_k=top_k
    )

    print(
        f"Retrieved {len(retrieved_docs)} evidence chunks."
    )

    # --------------------------------------------------
    # 2. BUILD CONTEXT
    # --------------------------------------------------

    print("\n[2/6] Building grounded context...")

    context = build_context(
        retrieved_docs
    )

    # --------------------------------------------------
    # 3. GENERATE ANSWER
    # --------------------------------------------------

    print("\n[3/6] Generating grounded answer...")

    answer = generate_grounded_answer(
        question,
        context
    )

    print("\nGenerated Answer:")
    print("-" * 80)
    print(answer)
    print("-" * 80)

    # --------------------------------------------------
    # 4. CLAIM VERIFICATION
    # --------------------------------------------------

    print("\n[4/6] Verifying claims...")

    claim_verification = verify_claims_fast(
        answer,
        top_n=verification_top_n,
        batch_size=16
    )

    # --------------------------------------------------
    # 5. CONFIDENCE
    # --------------------------------------------------

    print("\n[5/6] Calculating confidence...")

    confidence_result = calculate_final_confidence(
        answer=answer,
        question=question,
        retrieved_docs=retrieved_docs,
        claim_verification=claim_verification
    )

    # --------------------------------------------------
    # 6. FINAL DECISION
    # --------------------------------------------------

    print("\n[6/6] Determining hallucination status...")

    final_decision = determine_final_decision(
        claim_verification=claim_verification,
        confidence_result=confidence_result
    )

    # --------------------------------------------------
    # SELF-CORRECTION
    # --------------------------------------------------

    if run_self_correction:

        self_correction_result = self_correct_answer(
            question=question,
            answer=answer,
            final_decision=final_decision,
            claim_verification=claim_verification
        )

    else:

        self_correction_result = {
            "correction_needed": False,
            "original_answer": answer,
            "corrected_answer": answer,
            "correction_applied": False
        }

    # --------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------

    pipeline_result = {

        "question": question,

        "retrieved_documents": retrieved_docs,

        "generated_answer": answer,

        "claim_verification": claim_verification,

        "confidence": confidence_result,

        "hallucination_detection": final_decision,

        "self_correction": self_correction_result
    }

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE")
    print("=" * 80)

    print(
        f"Final Decision: "
        f"{final_decision['decision']}"
    )

    print(
        f"Confidence: "
        f"{confidence_result['confidence_score']:.4f} "
        f"({confidence_result['confidence_level']})"
    )

    print(
        f"Claim Consistency: "
        f"{claim_verification['consistency_score']:.4f}"
    )

    print("=" * 80)

    return pipeline_result

In [57]:
# Cell 24 — Final End-to-End Test

test_question = (
    "What are the risk factors for type 2 diabetes?"
)

final_pipeline_result = run_medical_rag_pipeline(
    question=test_question,
    top_k=10,
    verification_top_n=10,
    run_self_correction=True
)

MEDICAL RAG + HALLUCINATION DETECTION PIPELINE

Question: What are the risk factors for type 2 diabetes?

[1/6] Retrieving evidence...
Retrieved 10 evidence chunks.

[2/6] Building grounded context...

[3/6] Generating grounded answer...
Attempt 1/3 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 304.618467ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailu

RuntimeError: Gemini generation failed after 3 attempts. Last error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 52.073561956s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimension